# Stable Diffusion: Theory & Practice

> **Course:** Computer Vision — Generative Models Module  
> **Prerequisites:** Familiarity with PyTorch, basic probability, and neural networks  
> **Runtime:** GPU recommended (T4 or better on Colab)

---

## Table of Contents

1. [Background: From VAEs to Diffusion](#1-background)
2. [Diffusion Models: The Core Idea](#2-diffusion-theory)
3. [Latent Diffusion Models (LDM)](#3-ldm)
4. [The U-Net Denoiser](#4-unet)
5. [CLIP Text Conditioning](#5-clip)
6. [The Sampling Process](#6-sampling)
7. [Environment Setup](#7-setup)
8. [Practical 1 — Text-to-Image](#8-text2img)
9. [Practical 2 — Image-to-Image](#9-img2img)
10. [Practical 3 — Inpainting](#10-inpainting)
11. [Practical 4 — Prompt Engineering](#11-prompts)
12. [Practical 5 — Visualizing the Denoising Process](#12-denoising-viz)
13. [Advanced: LoRA and Fine-tuning Concepts](#13-lora)
14. [Exercises](#14-exercises)

---
<a name='1-background'></a>
## 1. Background: From VAEs to Diffusion

### Why generative models?

Generative models learn the **data distribution** $p(x)$ and can sample new instances from it. The main families are:

| Model | Core Idea | Strengths | Weaknesses |
|---|---|---|---|
| **VAE** | Encode to latent → decode | Stable training, continuous latent | Blurry outputs |
| **GAN** | Generator vs. Discriminator | Sharp images | Mode collapse, training instability |
| **Flow** | Invertible transformations | Exact likelihood | Expensive architectures |
| **Diffusion** | Iterative denoising | State-of-the-art quality | Slow sampling (improving) |

### Variational Autoencoders (Brief Recap)

A VAE learns:
- An **encoder** $q_\phi(z|x)$: maps image $x$ to latent distribution $(\mu, \sigma)$
- A **decoder** $p_\theta(x|z)$: reconstructs from sampled latent $z \sim \mathcal{N}(\mu, \sigma^2)$

The ELBO loss:
$$\mathcal{L}_{\text{VAE}} = \underbrace{\mathbb{E}[\log p_\theta(x|z)]}_{\text{reconstruction}} - \underbrace{D_{KL}(q_\phi(z|x) \| p(z))}_{\text{regularization}}$$

**Key takeaway:** VAEs provide a compressed, continuous latent space — a property that Stable Diffusion *reuses* to make diffusion tractable.

---
<a name='2-diffusion-theory'></a>
## 2. Diffusion Models: The Core Idea

Diffusion models are inspired by **non-equilibrium thermodynamics**. The key insight:

> If we know how to *destroy* data by gradually adding noise, we can learn to *reverse* this process and generate data from pure noise.

### 2.1 Forward Process (Noise Schedule)

Given a clean image $x_0$, we define a Markov chain that adds Gaussian noise over $T$ timesteps:

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t;\ \sqrt{1 - \beta_t}\, x_{t-1},\ \beta_t \mathbf{I})$$

Where $\{\beta_t\}$ is the **noise schedule** (typically from $\beta_1 \approx 10^{-4}$ to $\beta_T \approx 0.02$).

A beautiful property: we can sample **any** timestep directly:

$$q(x_t | x_0) = \mathcal{N}(x_t;\ \sqrt{\bar{\alpha}_t}\, x_0,\ (1 - \bar{\alpha}_t) \mathbf{I})$$

where $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$.

So explicitly: $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon$, with $\epsilon \sim \mathcal{N}(0, \mathbf{I})$.

### 2.2 Reverse Process (Denoising)

The reverse process is also Gaussian (for small $\beta_t$):

$$p_\theta(x_{t-1} | x_t) = \mathcal{N}(x_{t-1};\ \mu_\theta(x_t, t),\ \Sigma_\theta(x_t, t))$$

A neural network $\epsilon_\theta$ learns to **predict the noise** $\epsilon$ added to $x_0$.

### 2.3 Training Objective

The simplified DDPM loss (Ho et al., 2020):

$$\mathcal{L}_{\text{simple}} = \mathbb{E}_{t, x_0, \epsilon} \left[ \| \epsilon - \epsilon_\theta(\underbrace{\sqrt{\bar{\alpha}_t} x_0 + \sqrt{1-\bar{\alpha}_t}\epsilon}_{x_t},\ t) \|^2 \right]$$

**Translation:** At each training step:
1. Sample a clean image $x_0$ from the dataset
2. Sample a random timestep $t$ and noise $\epsilon$
3. Create noisy image $x_t$
4. Ask the network to predict $\epsilon$ from $x_t$ and $t$
5. Minimize prediction error

In [ ]:
# ── Visualizing the Forward Diffusion Process ──────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import requests
from io import BytesIO

def get_noise_schedule(T=1000, schedule='linear'):
    """Returns beta_t, alpha_t, alpha_bar_t for a given noise schedule."""
    if schedule == 'linear':
        betas = np.linspace(1e-4, 0.02, T)
    elif schedule == 'cosine':
        # Improved DDPM (Nichol & Dhariwal, 2021)
        s = 0.008
        steps = np.arange(T + 1)
        f = np.cos((steps / T + s) / (1 + s) * np.pi / 2) ** 2
        alphas_bar = f / f[0]
        betas = 1 - alphas_bar[1:] / alphas_bar[:-1]
        betas = np.clip(betas, 0, 0.999)
        alphas = 1 - betas
        alphas_bar = np.cumprod(alphas)
        return betas, alphas, alphas_bar
    alphas = 1 - betas
    alphas_bar = np.cumprod(alphas)
    return betas, alphas, alphas_bar

def forward_diffusion(x0_arr, t, alphas_bar):
    """Apply q(x_t | x_0) directly using the closed-form expression."""
    noise = np.random.randn(*x0_arr.shape)
    alpha_bar_t = alphas_bar[t]
    x_t = np.sqrt(alpha_bar_t) * x0_arr + np.sqrt(1 - alpha_bar_t) * noise
    return np.clip(x_t, 0, 1), noise

# Load a sample image
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"
try:
    resp = requests.get(url, timeout=5)
    img = Image.open(BytesIO(resp.content)).convert('RGB').resize((128, 128))
    x0 = np.array(img) / 255.0
except:
    # Fallback: generate a synthetic colorful image
    x0 = np.zeros((128, 128, 3))
    x0[:64, :64] = [0.9, 0.2, 0.2]   # red
    x0[:64, 64:] = [0.2, 0.7, 0.2]   # green
    x0[64:, :64] = [0.2, 0.2, 0.9]   # blue
    x0[64:, 64:] = [0.9, 0.8, 0.1]   # yellow

# ── Plot Forward Process ───────────────────────────────────────────────────
T = 1000
betas_lin, alphas_lin, abar_lin   = get_noise_schedule(T, 'linear')
betas_cos, alphas_cos, abar_cos   = get_noise_schedule(T, 'cosine')

timesteps_to_show = [0, 100, 250, 500, 750, 999]
fig, axes = plt.subplots(2, len(timesteps_to_show) + 1, figsize=(16, 5))
fig.suptitle('Forward Diffusion: Linear vs Cosine Schedule', fontsize=14, fontweight='bold')

for row, (abar, label) in enumerate([(abar_lin, 'Linear'), (abar_cos, 'Cosine')]):
    axes[row, 0].set_ylabel(label, fontsize=12, fontweight='bold')
    axes[row, 0].imshow(x0)
    axes[row, 0].set_title('x₀ (clean)')
    axes[row, 0].axis('off')
    for col, t in enumerate(timesteps_to_show):
        xt, _ = forward_diffusion(x0, t, abar)
        axes[row, col + 1].imshow(xt)
        axes[row, col + 1].set_title(f't={t}')
        axes[row, col + 1].axis('off')

plt.tight_layout()
plt.savefig('forward_diffusion.png', dpi=120, bbox_inches='tight')
plt.show()

# ── Plot Schedule Comparison ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
t_range = np.arange(T)
axes[0].plot(t_range, abar_lin, label='Linear', color='steelblue')
axes[0].plot(t_range, abar_cos, label='Cosine', color='coral', linestyle='--')
axes[0].set_xlabel('Timestep t'); axes[0].set_ylabel('ᾱₜ (signal retention)')
axes[0].set_title('Signal Retention α̅ₜ per Schedule'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(t_range, betas_lin, label='Linear β', color='steelblue')
axes[1].plot(t_range, betas_cos, label='Cosine β', color='coral', linestyle='--')
axes[1].set_xlabel('Timestep t'); axes[1].set_ylabel('βₜ (noise added)')
axes[1].set_title('Noise Schedule βₜ'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📌 Observation: The cosine schedule adds noise more gradually at the start,")
print("   preserving low-frequency structure longer — leading to better sample quality.")

---
<a name='3-ldm'></a>
## 3. Latent Diffusion Models (LDM)

### The Scalability Problem

Running diffusion **directly in pixel space** for a 512×512 image means operating on a 786,432-dimensional vector. This is:
- Computationally expensive
- Redundant (most of image information is low-frequency)

### The LDM Solution (Rombach et al., 2022)

Stable Diffusion decouples the process into two stages:

```
┌─────────────────────────────────────────────────────────────────┐
│  Stage 1: Perceptual Compression (trained once, frozen)         │
│  Image x (512×512×3) ──[VAE Encoder]──► z (64×64×4)            │
│                         ◄[VAE Decoder]── z                      │
│                                                                 │
│  Stage 2: Latent Diffusion (the generative model)               │
│  z₀ ──[Forward]──► zₜ (noisy latent)                           │
│  zₜ + condition c ──[U-Net ε_θ]──► z₀ (denoised latent)        │
│  z₀ ──[VAE Decoder]──► x̂ (generated image)                    │
└─────────────────────────────────────────────────────────────────┘
```

**Compression ratio:** 512×512×3 → 64×64×4, a **48× reduction** in dimensionality!

### Why does this work?

The VAE encoder learns a **perceptually meaningful** compression — it preserves semantic content while discarding imperceptible high-frequency noise. The diffusion model then operates in this *semantic* latent space, making learning easier and inference faster.

### Key Parameters in SD

| Component | Detail |
|---|---|
| VAE latent channels | 4 |
| Downsampling factor | 8× (per spatial dim) |
| Latent resolution (512 input) | 64×64 |
| Diffusion timesteps T | 1000 (training) |
| Inference steps | 20–50 (with fast samplers) |
| U-Net parameters | ~860M |
| CLIP text encoder | ~340M |

---
<a name='4-unet'></a>
## 4. The U-Net Denoiser Architecture

The denoising network $\epsilon_\theta(z_t, t, c)$ is a **conditional U-Net** with:

### 4.1 Timestep Embedding

Timestep $t$ is encoded via **sinusoidal positional embeddings** (same as Transformers):

$$\text{emb}(t)_i = \begin{cases} \sin(t / 10000^{i/d}) & \text{if } i \text{ even} \\ \cos(t / 10000^{(i-1)/d}) & \text{if } i \text{ odd} \end{cases}$$

Then projected through 2 linear layers into a feature vector added to each ResNet block.

### 4.2 Architecture Overview

```
Input z_t (64×64×4)
        │
   [Conv 3×3] ─────────────────────────────────────────────────────────────┐
        │                                                                   │
  ┌─────▼──────┐                                                     ┌─────▼──────┐
  │ DownBlock 1│ (ResNet + SelfAttn)  ──────── skip ──────────────► │  UpBlock 1 │
  └─────┬──────┘                                                     └─────┬──────┘
  ┌─────▼──────┐                                                     ┌─────▼──────┐
  │ DownBlock 2│ (ResNet + CrossAttn) ─────── skip ──────────────►  │  UpBlock 2 │
  └─────┬──────┘                                                     └─────┬──────┘
  ┌─────▼──────┐                                                     ┌─────▼──────┐
  │ DownBlock 3│ (ResNet + CrossAttn) ─────── skip ──────────────►  │  UpBlock 3 │
  └─────┬──────┘                                                     └─────┬──────┘
        │
  ┌─────▼──────┐
  │  MidBlock  │ (ResNet + SelfAttn + ResNet)
  └────────────┘
        │
   [Conv 3×3] → predicted noise ε̂ (64×64×4)
```

### 4.3 Cross-Attention for Text Conditioning

At each attention block:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d}}\right) V$$

- **Q** (Query): from spatial features of $z_t$  
- **K, V** (Key, Value): from text embeddings $c = \text{CLIP}(\text{prompt})$

This is how the text "controls" the denoising — at every layer, the network attends to the relevant parts of the text embedding.

In [ ]:
# ── Minimal U-Net building blocks (educational, not SD's actual code) ──────
import torch
import torch.nn as nn
import math

class SinusoidalPositionEmbedding(nn.Module):
    """Encodes the diffusion timestep t into a vector."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t[:, None].float() * emb[None, :]      # [B, half_dim]
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)  # [B, dim]
        return emb

class ResBlock(nn.Module):
    """ResNet block with timestep conditioning."""
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1  = nn.Sequential(nn.GroupNorm(8, in_ch), nn.SiLU(), nn.Conv2d(in_ch, out_ch, 3, padding=1))
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, out_ch))
        self.conv2  = nn.Sequential(nn.GroupNorm(8, out_ch), nn.SiLU(), nn.Conv2d(out_ch, out_ch, 3, padding=1))
        self.residual = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(x)
        h = h + self.time_mlp(t_emb)[:, :, None, None]  # broadcast over H, W
        h = self.conv2(h)
        return h + self.residual(x)

class CrossAttention(nn.Module):
    """Cross-attention: queries from image features, keys/values from text."""
    def __init__(self, query_dim, context_dim, heads=8):
        super().__init__()
        self.heads = heads
        self.scale = (query_dim // heads) ** -0.5
        self.to_q = nn.Linear(query_dim, query_dim, bias=False)
        self.to_k = nn.Linear(context_dim, query_dim, bias=False)
        self.to_v = nn.Linear(context_dim, query_dim, bias=False)
        self.to_out = nn.Linear(query_dim, query_dim)

    def forward(self, x, context):
        B, N, C = x.shape
        H = self.heads
        q = self.to_q(x).reshape(B, N, H, C // H).transpose(1, 2)
        k = self.to_k(context).reshape(B, -1, H, C // H).transpose(1, 2)
        v = self.to_v(context).reshape(B, -1, H, C // H).transpose(1, 2)
        attn = torch.softmax(torch.matmul(q, k.transpose(-2, -1)) * self.scale, dim=-1)
        out = torch.matmul(attn, v).transpose(1, 2).reshape(B, N, C)
        return self.to_out(out)

# ── Demonstrate timestep embedding ────────────────────────────────────────
emb_module = SinusoidalPositionEmbedding(dim=256)
timesteps = torch.tensor([0, 100, 500, 999])
embeddings = emb_module(timesteps)  # [4, 256]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
im = axes[0].imshow(embeddings.detach().numpy(), aspect='auto', cmap='RdBu_r')
axes[0].set_title('Sinusoidal Timestep Embeddings (256-dim)\neach row = one timestep')
axes[0].set_xlabel('Embedding Dimension')
axes[0].set_yticks([0,1,2,3]); axes[0].set_yticklabels(['t=0','t=100','t=500','t=999'])
plt.colorbar(im, ax=axes[0])

axes[1].plot(embeddings[0].detach(), label='t=0', alpha=0.8)
axes[1].plot(embeddings[1].detach(), label='t=100', alpha=0.8)
axes[1].plot(embeddings[2].detach(), label='t=500', alpha=0.8)
axes[1].plot(embeddings[3].detach(), label='t=999', alpha=0.8)
axes[1].set_title('Embedding Values per Timestep')
axes[1].set_xlabel('Dimension'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("Embedding shape:", embeddings.shape, "\nEach timestep has a unique signature for the network.")

---
<a name='5-clip'></a>
## 5. CLIP Text Conditioning

### What is CLIP?

**Contrastive Language-Image Pretraining** (Radford et al., 2021) trains two encoders jointly:
- A **text encoder** (Transformer)
- An **image encoder** (ViT or ResNet)

...to maximize similarity of matching image-text pairs in a shared embedding space.

### How Stable Diffusion Uses CLIP

1. Your prompt → **CLIP Tokenizer** → token IDs (max 77 tokens)
2. Token IDs → **CLIP Text Transformer** → text embeddings $c \in \mathbb{R}^{77 \times 768}$
3. Embeddings $c$ are passed as **Key** and **Value** in every cross-attention layer of the U-Net

### Classifier-Free Guidance (CFG)

To strengthen adherence to the prompt, SD uses **Classifier-Free Guidance**:

$$\hat{\epsilon}_\theta(z_t, c) = \epsilon_\theta(z_t, \varnothing) + w \cdot (\epsilon_\theta(z_t, c) - \epsilon_\theta(z_t, \varnothing))$$

Where:
- $\varnothing$ = unconditional embedding (empty prompt or `""`)  
- $c$ = text embedding of your prompt  
- $w$ = **guidance scale** (typically 7.5)

**Interpretation:** We move the prediction *away* from "no prompt" and *toward* the target prompt, scaled by $w$. Higher $w$ → stronger prompt adherence but less diversity.

| Guidance Scale | Effect |
|---|---|
| 1.0 | No guidance (pure model prior) |
| 3–5 | Loose adherence, creative |
| 7–9 | Balanced (default) |
| 12–20 | Strong adherence, potential artifacts |

---
<a name='6-sampling'></a>
## 6. The Sampling Process — DDIM & PNDM

### DDPM Sampling (Slow)

The original sampling from Ho et al. requires all $T=1000$ reverse steps:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}}\epsilon_\theta(x_t, t)\right) + \sigma_t z$$

### DDIM Sampling (Fast, Deterministic)

Denoising Diffusion Implicit Models (Song et al., 2020) reformulate as a non-Markovian process, allowing **arbitrary step skipping**:

$$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} \underbrace{\left(\frac{x_t - \sqrt{1-\bar{\alpha}_t}\, \epsilon_\theta}{\sqrt{\bar{\alpha}_t}}\right)}_{\text{predicted } x_0} + \underbrace{\sqrt{1-\bar{\alpha}_{t-1} - \sigma_t^2}\, \epsilon_\theta}_{\text{direction to } x_t} + \underbrace{\sigma_t z}_{\text{noise}}$$

Setting $\sigma_t = 0$ makes sampling **fully deterministic** — same seed → same image.

### Sampler Comparison

| Sampler | Steps | Deterministic | Notes |
|---|---|---|---|
| DDPM | 1000 | No | Original, slow |
| DDIM | 20–50 | Yes | Fast, good quality |
| PNDM | 20–50 | Yes | Better quality than DDIM |
| DPM-Solver++ | 10–20 | Yes | State-of-the-art speed/quality |
| Euler Ancestral | 20–30 | No | Creative, varied outputs |

---
<a name='7-setup'></a>
## 7. Environment Setup

We use Hugging Face `diffusers` — the standard library for working with diffusion models.

> ⚠️ **GPU Required:** Go to **Runtime → Change runtime type → T4 GPU** before proceeding.

In [ ]:
# ── Install dependencies ────────────────────────────────────────────────────
!pip install -q diffusers transformers accelerate safetensors xformers

import torch
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if device == 'cuda' else torch.float32

print(f"✅ Device : {device}")
if device == 'cuda':
    print(f"   GPU    : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. Some cells will be slow or skipped.")

In [ ]:
# ── Load SD 1.5 pipeline ────────────────────────────────────────────────────
# This downloads ~2.5 GB of model weights on first run and caches them.
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

MODEL_ID = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    safety_checker=None,    # disable for educational use
)

# Swap to a fast scheduler
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)

# Memory optimizations for Colab's T4
if device == 'cuda':
    pipe.enable_attention_slicing()   # trades speed for VRAM
    try:
        pipe.enable_xformers_memory_efficient_attention()
        print("✅ xFormers attention enabled")
    except Exception:
        print("ℹ️  xFormers not available, using default attention")

print("\n✅ Pipeline loaded successfully!")
print(f"   Scheduler : {pipe.scheduler.__class__.__name__}")
print(f"   VAE       : {pipe.vae.__class__.__name__}")
print(f"   U-Net     : {sum(p.numel() for p in pipe.unet.parameters()) / 1e6:.0f}M params")
print(f"   CLIP      : {sum(p.numel() for p in pipe.text_encoder.parameters()) / 1e6:.0f}M params")

---
<a name='8-text2img'></a>
## 8. Practical 1 — Text-to-Image Generation

The simplest use case: a text prompt → a generated image.

**Anatomy of a call:**
```
pipe(
    prompt            → positive text description
    negative_prompt   → what to avoid
    num_inference_steps → denoising steps (quality vs speed)
    guidance_scale    → CFG weight w
    generator         → torch.Generator for reproducibility
    height / width    → output resolution (must be multiples of 8)
)
```

In [ ]:
import torch
import matplotlib.pyplot as plt
from PIL import Image # Added for explicit Image conversion
import numpy as np # Added for type checking

def show_images(images, titles=None, cols=4, figsize=None):
    n = len(images)
    rows = (n + cols - 1) // cols
    figsize = figsize or (cols * 3.5, rows * 3.5)
    fig, axes = plt.subplots(rows, cols, figsize=figsize)

    # Ensure axes is a flat iterable of AxesSubplot objects
    if isinstance(axes, np.ndarray):
        axes = axes.flatten()
    elif isinstance(axes, plt.Axes): # Single axis object case (rows=1, cols=1)
        axes = [axes]
    else:
        axes = [] # Should not happen, but for robustness

    # If fewer images than subplots, use only the necessary axes
    if n < len(axes):
        axes = axes[:n]

    for i, (ax, img) in enumerate(zip(axes, images)):
        # Ensure img is a PIL Image object before displaying
        if isinstance(img, np.ndarray):
            # Assuming image data is in [0, 1] range for float arrays
            if img.dtype == np.float32 or img.dtype == np.float64:
                img = (img * 255).astype(np.uint8)
            img = Image.fromarray(img)
        ax.imshow(img)
        ax.axis('off')
        if titles: ax.set_title(titles[i], fontsize=8)

    # Turn off any remaining empty axes if n < len(axes_all)
    # (axes_all refers to the original full set of axes from plt.subplots)
    # This part needs to be careful not to re-iterate on the potentially sliced 'axes'
    # Let's ensure all axes created by subplots are turned off if not used
    all_axes_from_subplots = plt.gcf().get_axes() # Get all axes from the current figure
    for ax_full in all_axes_from_subplots:
        if ax_full not in axes: # If an axis is not used for an image
            ax_full.axis('off')

    plt.tight_layout()
    plt.show()

SEED = 42
generator = torch.Generator(device=device).manual_seed(SEED)

prompt = "a futuristic city at sunset, cyberpunk style, neon lights, ultra-detailed, 4k"
negative_prompt = "blurry, low quality, distorted, watermark, text"

with torch.inference_mode():
    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=25,
        guidance_scale=7.5,
        generator=generator,
        height=512, width=512,
    )

image = result.images[0]
show_images([image], titles=[f'Prompt: "{prompt[:60]}..."'])

In [ ]:
# ── Effect of Guidance Scale ─────────────────────────────────────────────────
# Demonstrates how CFG scale controls prompt adherence vs. diversity

prompt = "a majestic lion in a meadow, oil painting style"
guidance_scales = [1.0, 3.0, 7.5, 12.0, 20.0]
images, titles = [], []

for w in guidance_scales:
    g = torch.Generator(device=device).manual_seed(SEED)
    with torch.inference_mode():
        img = pipe(prompt=prompt, guidance_scale=w, num_inference_steps=25,
                   generator=g, height=512, width=512).images[0]
    images.append(img)
    titles.append(f'CFG = {w}')

show_images(images, titles=titles, cols=5, figsize=(18, 4))
print("\n📌 Notice: CFG=1 ignores the prompt; high CFG oversaturates/distorts.")

---
<a name='9-img2img'></a>
## 9. Practical 2 — Image-to-Image

### Theory

Instead of starting from pure noise $z_T \sim \mathcal{N}(0, I)$, we:
1. Encode the input image → latent $z_0$
2. Add noise only up to step $t^* = T \cdot \text{strength}$ (i.e., partial forward diffusion)
3. Run the reverse process from $z_{t^*}$ (not from $z_T$)

**`strength`** controls how much the output differs from the input:
- `strength=0.0` → identical to input
- `strength=1.0` → effectively text-to-image (fully redrawn)
- `strength=0.5-0.75` → style transfer / guided editing

In [ ]:
# ── Image-to-Image ──────────────────────────────────────────────────────────
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image
import requests
from io import BytesIO

# Load img2img pipeline (reuses weights from memory)
img2img_pipe = StableDiffusionImg2ImgPipeline(
    vae=pipe.vae, text_encoder=pipe.text_encoder,
    tokenizer=pipe.tokenizer, unet=pipe.unet,
    scheduler=pipe.scheduler, safety_checker=None,
    feature_extractor=None, requires_safety_checker=False
).to(device)

# Create or load an input image (simple colored rectangles as fallback)
from PIL import ImageDraw
init_img = Image.new('RGB', (512, 512), color=(100, 150, 200))
draw = ImageDraw.Draw(init_img)
draw.ellipse([100, 100, 412, 412], fill=(200, 100, 80))
draw.rectangle([180, 280, 332, 450], fill=(80, 160, 80))

strengths = [0.3, 0.5, 0.7, 0.9]
prompt = "a detailed oil painting of a landscape with mountains"
images = [init_img]
titles = ['Input Image']

for s in strengths:
    g = torch.Generator(device=device).manual_seed(SEED)
    with torch.inference_mode():
        out = img2img_pipe(
            prompt=prompt, image=init_img,
            strength=s, guidance_scale=7.5,
            num_inference_steps=30, generator=g
        ).images[0]
    images.append(out)
    titles.append(f'strength={s}')

show_images(images, titles=titles, cols=5, figsize=(18, 4))
print("\n📌 Larger strength → more creative deviation from input.")

---
<a name='10-inpainting'></a>
## 10. Practical 3 — Inpainting

### Theory

Inpainting generates content for a **masked region** while keeping the rest intact:

1. The mask (white = regenerate, black = keep) is passed to the pipeline
2. The U-Net receives a **9-channel input**: original 4 latent channels + 4 masked latent channels + 1 mask channel
3. Only the masked region participates in diffusion; unmasked regions are stitched back

This is used for object removal, replacement, and image editing.

In [ ]:
# ── Inpainting ──────────────────────────────────────────────────────────────
from diffusers import StableDiffusionInpaintPipeline
import numpy as np

# For proper inpainting we need a separate inpainting model checkpoint
# NOTE: if VRAM is tight, comment the inpaint_pipe load and use the
# conceptual demo below.

# Create a synthetic scene + mask
scene = Image.new('RGB', (512, 512), (135, 206, 235))   # sky blue
draw = ImageDraw.Draw(scene)
draw.rectangle([0, 350, 512, 512], fill=(34, 139, 34))  # green ground
draw.ellipse([180, 150, 330, 300], fill=(255, 215, 0))  # yellow sun

# Mask: white over the sun region → will be inpainted
mask = Image.new('L', (512, 512), 0)  # black = keep
mask_draw = ImageDraw.Draw(mask)
mask_draw.ellipse([160, 130, 350, 320], fill=255)       # white = regenerate

# Show scene and mask
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(scene); axes[0].set_title('Original Scene'); axes[0].axis('off')
axes[1].imshow(mask, cmap='gray'); axes[1].set_title('Mask (white=regenerate)'); axes[1].axis('off')

# Composite: show what will be kept
composite = np.array(scene).copy()
composite[np.array(mask) > 128] = [200, 200, 200]
axes[2].imshow(composite); axes[2].set_title('Masked Input'); axes[2].axis('off')
plt.suptitle('Inpainting Setup', fontweight='bold'); plt.tight_layout(); plt.show()

print("\n📌 Inpainting concept demonstrated.")
print("   To run actual inpainting, load: 'runwayml/stable-diffusion-inpainting'")
print("   with StableDiffusionInpaintPipeline and pass (image=scene, mask_image=mask)")

---
<a name='11-prompts'></a>
## 11. Practical 4 — Prompt Engineering

Prompt crafting is an art. CLIP was trained on internet text, so it responds to photography and art vocabulary.

### Anatomy of a Good Prompt

```
[Subject] [Modifier] [Style] [Artist reference] [Quality tokens] [Technical specs]

Example:
"a medieval knight ← subject
 standing in a foggy forest, dramatic lighting ← modifiers
 digital art, concept art ← style
 by Greg Rutkowski ← artist reference
 highly detailed, sharp focus ← quality tokens
 8k, trending on ArtStation" ← technical specs
```

### Token Weighting

In many pipelines you can emphasize tokens:
- `(word)` — slight boost
- `(word:1.5)` — 50% more weight
- `[word]` — slight reduction

In [ ]:
# ── Prompt Comparison Experiment ────────────────────────────────────────────
experiments = [
    {
        'label': 'Minimal',
        'prompt': 'a robot'
    },
    {
        'label': 'Medium',
        'prompt': 'a friendly robot in a garden, colorful, detailed'
    },
    {
        'label': 'Rich',
        'prompt': 'a sleek futuristic robot standing in a lush japanese garden, cherry blossoms, soft morning light, highly detailed, photorealistic, 4k, trending on ArtStation'
    },
    {
        'label': 'With Negative',
        'prompt': 'a sleek futuristic robot standing in a lush japanese garden, cherry blossoms, soft morning light, highly detailed, photorealistic, 4k',
        'negative': 'ugly, deformed, low quality, blurry, watermark, extra limbs'
    },
]

results = []
for exp in experiments:
    g = torch.Generator(device=device).manual_seed(SEED)
    neg = exp.get('negative', '')
    with torch.inference_mode():
        img = pipe(
            prompt=exp['prompt'], negative_prompt=neg,
            num_inference_steps=25, guidance_scale=7.5,
            generator=g, height=512, width=512
        ).images[0]
    results.append((img, exp['label']))

show_images([r[0] for r in results], titles=[r[1] for r in results], cols=4, figsize=(16, 4))
print("\n📌 Richer prompts + negative prompts consistently produce higher quality outputs.")

---
<a name='12-denoising-viz'></a>
## 12. Practical 5 — Visualizing the Denoising Process

We hook into the pipeline's callback mechanism to capture intermediate latents at each denoising step and decode them to images. This reveals how SD builds up structure from noise.

In [ ]:
# ── Denoising Trajectory Visualization ─────────────────────────────────────
import torch
from diffusers import StableDiffusionPipeline
import numpy as np

intermediates = []
CAPTURE_EVERY = 5  # capture every N steps

def latent_to_image(latents, vae):
    """Decode latent tensor to PIL image."""
    latents = latents / vae.config.scaling_factor
    with torch.inference_mode():
        decoded = vae.decode(latents).sample
    decoded = (decoded / 2 + 0.5).clamp(0, 1)
    decoded = decoded.cpu().permute(0, 2, 3, 1).numpy()
    return Image.fromarray((decoded[0] * 255).astype(np.uint8))

def capture_callback(pipe, step, timestep, callback_kwargs):
    if step % CAPTURE_EVERY == 0 or step == 0:
        lat = callback_kwargs['latents'].clone()
        img = latent_to_image(lat, pipe.vae)
        intermediates.append((step, timestep.item(), img))
    return callback_kwargs

intermediates.clear()
prompt = "a majestic eagle soaring over snowy mountains, epic landscape photography"
g = torch.Generator(device=device).manual_seed(7)

with torch.inference_mode():
    final = pipe(
        prompt=prompt,
        negative_prompt="blurry, low quality",
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=g,
        callback_on_step_end=capture_callback,
        callback_on_step_end_tensor_inputs=['latents'],
    ).images[0]

# Add final image
intermediates.append((30, 0, final))

# Plot trajectory
imgs   = [x[2] for x in intermediates]
labels = [f'Step {x[0]}\nt={x[1]}' for x in intermediates]

cols = min(len(imgs), 7)
rows = (len(imgs) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
axes = axes.flatten() if len(imgs) > 1 else [axes]

for ax, img, label in zip(axes, imgs, labels):
    ax.imshow(img); ax.set_title(label, fontsize=8); ax.axis('off')
for ax in axes[len(imgs):]: ax.axis('off')

plt.suptitle('Denoising Trajectory: Noise → Structure → Detail',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('denoising_trajectory.png', dpi=120, bbox_inches='tight')
plt.show()
print("\n📌 Early steps build global structure; later steps add fine details.")

---
<a name='13-lora'></a>
## 13. Advanced: LoRA and Fine-tuning Concepts

### Why Fine-tune?

Base SD knows millions of concepts, but:
- Can't generate **your specific face** or a **custom product**
- May lack knowledge of **niche artistic styles**
- Prompt-engineering has limits

### LoRA: Low-Rank Adaptation

Full fine-tuning of 860M U-Net params is expensive. LoRA (Hu et al., 2021) decomposes weight updates:

$$W_{\text{new}} = W_0 + \Delta W = W_0 + \underbrace{B \cdot A}_{\text{rank-}r\text{ decomposition}}$$

Where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$, with $r \ll \min(d, k)$ (typically $r=4$ to $r=64$).

**Result:** Instead of updating $d \times k$ params, only $r(d+k)$ — a **100–1000× reduction**.

### Fine-tuning Methods

| Method | Data needed | Time | Params trained | Use case |
|---|---|---|---|---|
| **DreamBooth** | 5–30 images | ~1h GPU | ~860M (full) | Specific subject/person |
| **LoRA** | 5–100 images | 30min GPU | ~3M | Style, concept |
| **Textual Inversion** | 5–20 images | 1-2h GPU | ~768 (embedding) | New token for concept |

In [ ]:
# ── Load a community LoRA from Hugging Face Hub ─────────────────────────────
# LoRAs are small weight delta files (~5-50 MB) that modify the U-Net/text encoder.
# We demonstrate loading one without full fine-tuning.

# Reset pipeline to base weights first
pipe.unload_lora_weights()

# Many LoRAs are available at civitai.com or huggingface.co.
# Here we show the API — uncomment and substitute a real LoRA path:
# pipe.load_lora_weights("sayakpaul/sd-model-finetuned-lora-t4", weight_name="pytorch_lora_weights.bin")

# ── LoRA weight structure (educational) ────────────────────────────────────
import torch.nn as nn

class LoRALayer(nn.Module):
    """Minimal LoRA: adds a low-rank branch to an existing linear layer."""
    def __init__(self, original: nn.Linear, rank: int = 4, alpha: float = 1.0):
        super().__init__()
        self.original = original
        self.rank     = rank
        self.scale    = alpha / rank
        d_out, d_in  = original.weight.shape
        # A: sampled from N(0, 1/r),  B: zeros (so Δ=0 at start)
        self.lora_A  = nn.Parameter(torch.randn(rank, d_in) * (1 / rank))
        self.lora_B  = nn.Parameter(torch.zeros(d_out, rank))

    def forward(self, x):
        orig_out = self.original(x)          # W₀ · x
        lora_out = x @ self.lora_A.T @ self.lora_B.T  # (B·A) · x
        return orig_out + self.scale * lora_out

# Show parameter count comparison
d_in, d_out, rank = 768, 768, 8
full_params = d_in * d_out
lora_params = rank * d_in + rank * d_out
print(f"Full linear layer params : {full_params:,}")
print(f"LoRA (rank={rank}) params : {lora_params:,}")
print(f"Compression ratio        : {full_params / lora_params:.1f}×")

# Visualize rank-decomposition concept
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
W0 = np.random.randn(64, 64)
A  = np.random.randn(rank, 64) * 0.1
B  = np.zeros((64, rank))
dW = B @ A
axes[0].imshow(W0,      cmap='RdBu_r', vmin=-2, vmax=2); axes[0].set_title('W₀ (frozen, 64×64)'); axes[0].axis('off')
axes[1].imshow(dW,      cmap='RdBu_r', vmin=-0.1, vmax=0.1); axes[1].set_title(f'ΔW = B·A (rank={rank})\ninitially zeros'); axes[1].axis('off')
axes[2].imshow(W0 + dW, cmap='RdBu_r', vmin=-2, vmax=2); axes[2].set_title('W₀ + ΔW (effective weight)'); axes[2].axis('off')
plt.suptitle('LoRA Weight Decomposition', fontweight='bold')
plt.tight_layout(); plt.show()

---
<a name='14-exercises'></a>
## 14. Exercises

### 🟢 Beginner

**Exercise 1 — Prompt Ablation**  
Pick a subject and generate 6 images with progressively richer prompts (bare noun → full richly-described prompt + negative). Create a figure comparing quality.

**Exercise 2 — Scheduler Comparison**  
Replace the scheduler with: `DDIMScheduler`, `PNDMScheduler`, `EulerAncestralDiscreteScheduler`, `DPMSolverMultistepScheduler`. Fix seed and prompt. Show all outputs with step counts and measure generation time.

```python
from diffusers import DDIMScheduler, PNDMScheduler, EulerAncestralDiscreteScheduler
# Your code here
```

---

### 🟡 Intermediate

**Exercise 3 — Noise Schedule from Scratch**  
Implement the `forward_diffusion` function and apply it to an MNIST digit. Plot the signal-to-noise ratio (SNR = $\bar{\alpha}_t / (1 - \bar{\alpha}_t)$) in dB for both linear and cosine schedules.

**Exercise 4 — DDIM Sampler**  
Implement DDIM sampling manually using the formula from Section 6, operating on random latents and using the pipeline's U-Net. Compare 10, 25, 50 step outputs.

---

### 🔴 Advanced

**Exercise 5 — Attention Map Visualization**  
Register forward hooks on the U-Net's cross-attention layers. For a given prompt, capture and visualize the attention maps for individual tokens (e.g., show which image regions attend to "mountain", "sky", "eagle").

**Exercise 6 — Minimal DDPM Training**  
Train a tiny DDPM on MNIST (or Fashion-MNIST) from scratch using the `ResBlock` and `SinusoidalPositionEmbedding` classes from Section 4. Show the training loss curve and generated samples every 5 epochs.

```python
# Skeleton:
class TinyUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.time_mlp = SinusoidalPositionEmbedding(128)
        # TODO: add encoder, bottleneck, decoder

    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        # TODO: forward pass
        pass

def training_loop(model, dataloader, optimizer, T=1000, epochs=50):
    betas, alphas, alphas_bar = get_noise_schedule(T)
    ab = torch.from_numpy(alphas_bar).float().to(device)
    for epoch in range(epochs):
        for batch, _ in dataloader:
            x0 = batch.to(device)
            t  = torch.randint(0, T, (x0.shape[0],), device=device)
            eps = torch.randn_like(x0)
            xt = torch.sqrt(ab[t])[:,None,None,None] * x0 + \
                 torch.sqrt(1 - ab[t])[:,None,None,None] * eps
            pred_eps = model(xt, t)
            loss = F.mse_loss(pred_eps, eps)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
```

---

## References

1. Ho et al. (2020). *Denoising Diffusion Probabilistic Models*. [arXiv:2006.11239](https://arxiv.org/abs/2006.11239)
2. Song et al. (2020). *Denoising Diffusion Implicit Models*. [arXiv:2010.02502](https://arxiv.org/abs/2010.02502)
3. Rombach et al. (2022). *High-Resolution Image Synthesis with Latent Diffusion Models*. [arXiv:2112.10752](https://arxiv.org/abs/2112.10752)
4. Radford et al. (2021). *Learning Transferable Visual Models From Natural Language Supervision* (CLIP). [arXiv:2103.00020](https://arxiv.org/abs/2103.00020)
5. Ho & Salimans (2022). *Classifier-Free Diffusion Guidance*. [arXiv:2207.12598](https://arxiv.org/abs/2207.12598)
6. Hu et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models*. [arXiv:2106.09685](https://arxiv.org/abs/2106.09685)
7. Nichol & Dhariwal (2021). *Improved Denoising Diffusion Probabilistic Models*. [arXiv:2102.09672](https://arxiv.org/abs/2102.09672)
8. 🤗 Diffusers Documentation: [huggingface.co/docs/diffusers](https://huggingface.co/docs/diffusers)
9. An Introduction to
Variational Autoencoders : [arXiv:1906.02691](https://arxiv.org/pdf/1906.02691)
10. Generative Adversarial Nets (GANS): [arxiv:1406.2661](https://arxiv.org/pdf/1406.2661)
